# Projet de Machine Learning 

### Sujet 1

**Contexte** :  Ce projet vise à mettre en place un système de maintenance prédictive pour les machines de l'usine et à analyser les causes qui font que la condition de la valve d'un cycle de production est non optimale parfois. 

**Objectif** : Construisez un modèle ML pour prédire si la condition valve est optimale (=100%) ou non, pour chaque cycle. Vous utiliserez les 2000 premiers cycles pour construire le modèle et le reste comme échantillon de test final.   

**Données** : Les données sont disponibles dans un Drive. Les données sont décrites à utiliser uniquement les 3 fichiers suivants, chaque ligne représentant un cycle :  
- PS2 (Pression (bar) échantillonnage 100Hz)  
- FS1 (Volume flow (l/min) échantillonnage 10Hz)  
- Profile : Fichier avec les variables dont la "valve condition" qui nous intéresse. 

## 1. Prétraitement des données

#### 1.1. Chargement des données et statistiques descriptives. 

In [6]:
# Imports, paramètres et chargement
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'data'     # dossier contenant PS2.txt, FS1.txt et profile.txt
N_DEV = 2000          # 2000 premiers cycles : construction du modèle
RANDOM_STATE = 42

ps2 = pd.read_csv(f'{DATA_DIR}/PS2.txt', sep=r'\s+', header=None)
fs1 = pd.read_csv(f'{DATA_DIR}/FS1.txt', sep=r'\s+', header=None)
profile = pd.read_csv(f'{DATA_DIR}/profile.txt', sep=r'\s+', header=None, names=['Cooler', 'Valve', 'Pump_leakage', 'Accumulator', 'Stable_flag'])

print('PS2     :', ps2.shape, '(cycles x mesures à 100 Hz)')
print('FS1     :', fs1.shape, '(cycles x mesures à 10 Hz)')
print('profile :', profile.shape)
assert len(ps2) == len(fs1) == len(profile), 'Nombre de cycles incohérent entre les fichiers'
profile.head()

ImportError: DLL load failed while importing algos: Une stratégie de contrôle d’application a bloqué ce fichier.

In [ ]:
# Séparation : 2000 premiers cycles (développement) / cycles restants (test final)
ps2_dev, ps2_test = ps2.iloc[:N_DEV], ps2.iloc[N_DEV:]
fs1_dev, fs1_test = fs1.iloc[:N_DEV], fs1.iloc[N_DEV:]
profile_dev, profile_test = profile.iloc[:N_DEV], profile.iloc[N_DEV:]

print('Cycles de développement :', len(profile_dev))
print('Cycles de test final    :', len(profile_test))

In [ ]:
# Statistiques descriptives (sur les 2000 cycles de développement)
print('--- profile.txt (conditions des composants) ---')
print(profile_dev.describe().round(2))

print('\n--- Signaux bruts ---')
for nom, df in [('PS2 (bar)', ps2_dev), ('FS1 (l/min)', fs1_dev)]:
    v = df.to_numpy().ravel()
    print(f'{nom:12s} min={v.min():8.2f} | moyenne={v.mean():8.2f} | médiane={np.median(v):8.2f} | '
          f'max={v.max():8.2f} | écart-type={v.std():7.2f}')

In [ ]:
# Allure moyenne d'un cycle selon la condition de la valve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (nom, df, hz) in zip(axes, [('PS2 – Pression (bar)', ps2_dev, 100), ('FS1 – Débit (l/min)', fs1_dev, 10)]):
    t = np.arange(df.shape[1]) / hz
    for val in sorted(profile_dev['Valve'].unique(), reverse=True):
        masque = (profile_dev['Valve'] == val).to_numpy()
        ax.plot(t, df[masque].mean(axis=0), label=f'Valve = {val} %')
    ax.set_title(nom + ' – moyenne par condition de valve')
    ax.set_xlabel('Temps dans le cycle (s)')
    ax.legend()
plt.tight_layout()
plt.show()

#### 1.2. Gestion des valeurs manquantes et détection des valeurs aberrantes.

## 2. Modélisation et évaluation  